# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hardikkk-1209/ML_Pipeline/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Unit of analysis: One row represents one content page for one client on one report date in the warehouse performance table.

Time window: I will use March 2026 (month=2026-03) as the development month for this contract. I will treat June 2026 as the final/sealed month and will not use it to develop label logic.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Features

- `impressions_90d` — historical search visibility available before the decision.
- `clicks_90d` — historical search interactions available before the decision.
- `ctr` — calculated from previously observed impressions and clicks.
- `avg_position` — historical search ranking performance available before the decision.
- `sessions_90d` — historical traffic observed before the decision.

### Label / proxy

- `trend_direction` — outcome/proxy representing the observed performance movement of a content page.

### Context

- `content_id` — identifies the content page being analyzed.
- `client_id` — identifies the client associated with the page.
- `content_type` — describes the type of content.
- `main_intent` — provides search-intent context for interpreting page performance.

### Excluded

- Future outcome information — excluded because it would not be available at the decision moment.
- Label-derived fields — excluded because they would leak information about the outcome into the features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# ============================================================
# SECTION 3 — VERIFY THE DATA CONTRACT
# ============================================================

# ============================================================
# QUERY 1 — GRAIN
# ============================================================

print("=== QUERY 1: GRAIN ===")

grain_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT
            report_date || '|' || client_hash_id || '|' || content_hash_id
        ) AS distinct_grain_rows
    FROM march
""").df()

display(grain_check)

print(
    "Expected grain: one row per report_date + client_hash_id + content_hash_id."
)

# ============================================================
# QUERY 2 — MARCH 2026 SLICE
# ============================================================

print("\n=== QUERY 2: MARCH 2026 SLICE ===")

slice_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS minimum_report_date,
        MAX(report_date) AS maximum_report_date
    FROM march
""").df()

display(slice_check)

# ============================================================
# QUERY 3 — AVAILABILITY
# ============================================================

print("\n=== QUERY 3: AVAILABILITY ===")

availability_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS available_rows,
        ROUND(
            100.0 *
            COUNT(*) FILTER (
                WHERE ga4_data_available IS TRUE
            ) / COUNT(*),
            2
        ) AS availability_percentage
    FROM march
""").df()

display(availability_check)

=== QUERY 1: GRAIN ===


,total_rows,distinct_grain_rows
0,9841378,9841378


Expected grain: one row per report_date + client_hash_id + content_hash_id.

=== QUERY 2: MARCH 2026 SLICE ===


,row_count,minimum_report_date,maximum_report_date
0,9841378,2026-03-01,2026-03-31



=== QUERY 3: AVAILABILITY ===


,total_rows,available_rows,availability_percentage
0,9841378,413966,4.21


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Data limits

This slice describes observed search and content-performance signals, but it does not provide a complete view of user behavior or business outcomes.

First, GA4 availability is limited in this March slice: only 413,966 of 9,841,378 rows (4.21%) have `ga4_data_available IS TRUE`. Therefore, conclusions based on GA4-derived signals should not be treated as representative of all content pages.

Second, the data is historical and observational. It can show associations between page-level signals and observed performance, but it cannot by itself establish that changing a feature or content property will cause a future performance change.

Finally, the June 2026 data is treated as a sealed final month, so I will not use it to develop the label or feature logic.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.